In [ ]:
!pip install numpy==1.26.4

In [ ]:
!pip install evaluate

In [ ]:
!pip install --upgrade transformers

In [ ]:
!pip install better_profanity

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Finetune RoBERTa for multi-label classification with LoRA
import pandas as pd
import numpy as np
from datasets import Dataset
from pathlib import Path
import os
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, PeftModel
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split


# Load Jigsaw dataset
jigsaw_dir = Path("/content/drive/My Drive/Jigsaw/train.csv")
df = pd.read_csv(jigsaw_dir)
# Define binary toxicity label (toxic if any of the categories are 1)
# df['label'] = (df[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']].sum(axis=1) > 0).astype(int)
label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
df["labels"] = df[label_cols].astype(float).values.tolist()
df = df[["comment_text", "labels"]]
df = df.rename(columns={"comment_text": "text"})

# Split
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Tokenizer & model
checkpoint = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize(example):
    return tokenizer(example['text'], truncation=True, padding='max_length', max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# Set format for Trainer
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

lora_config = LoraConfig(
    r=8,                   # rank
    lora_alpha=16,         # scaling
    target_modules=["query", "value"],  # attention projection layers in RoBERTa
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)

# Load model
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=len(label_cols),
    problem_type="multi_label_classification")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # shows % trainable
# Evaluation metrics
def compute_metrics(pred):
    logits, labels = pred
    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    preds = (probs >= 0.5).astype(int)
    f1_micro = f1_score(labels, preds, average="micro")
    f1_macro = f1_score(labels, preds, average="macro")
    return {"f1_micro": f1_micro, "f1_macro": f1_macro}

# Training configuration
training_args = TrainingArguments(
    output_dir="roberta-toxic-multilabel",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir="./logs",
    report_to=[]
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Train
trainer.train()
model_dir = Path("/content/drive/My Drive/DALI/model")
# Save model
model.save_pretrained(model_dir / "roberta-toxic-multilabel")
tokenizer.save_pretrained(model_dir / "roberta-toxic-multilabel")

# Evaluate
results = trainer.evaluate()
print("Evaluation:", results)


In [ ]:
# Test
from better_profanity import profanity
profanity.load_censor_words()

model_dir = Path("/content/drive/My Drive/DALI/model")
# Load manually labeled dataset
lyrics_path = Path('/content/drive/MyDrive/DALI/clean_explicit_comparison/lyrics_labeled.csv')
lyrics_df = pd.read_csv(lyrics_path).dropna(subset=['text'])
feature_df = lyrics_df['text']
true_pred = lyrics_df['label']
texts = feature_df.tolist()

model_lora_path = os.path.join("/content/drive/My Drive/DALI/model", 'roberta-toxic-multilabel')
checkpoint = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
base_model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=6,  # number of toxic categories
    problem_type="multi_label_classification"
)
model = PeftModel.from_pretrained(base_model, model_lora_path)
model.eval()
inputs = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
probs = torch.sigmoid(logits)

# Threshold to get binary predictions
threshold = 0.5
preds = (probs >= threshold).int()
preds_np = preds.cpu().numpy()                # convert to NumPy array
preds_multi = (preds_np.any(axis=1)).astype(int)  # 1 if any label positive else 0

In [1]:
# Compare the results
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score
def print_results(true, pred):
  print("Precision: ", precision_score(true, pred))
  print("Recall: ", recall_score(true, pred))
  print("Accuracy: ", accuracy_score(true, pred))
  print("F1: ", f1_score(true, pred))

In [ ]:
# Only searching bad words
search_pred = lyrics_df['text'].astype(str).apply(profanity.contains_profanity)
search_pred = search_pred.to_numpy().astype(int)
print_results(true_pred,search_pred)

In [ ]:
# With finetuned language model
search_multi_pred = np.logical_or(preds_multi,search_pred).astype(int)
print_results(true_pred,search_multi_pred)